# Twitter Sentiment Classification

Using `nltk.corpus.twitter_samples`, which ships 5,000 positive and 5,000 negative tweets. They're already labeled and clean enough to skip the usual scraping headache.

This is a textbook TF-IDF + logistic regression setup. The interesting part of doing sentiment on tweets is the tokenization — tweets have hashtags, @mentions, emoji, slang — so I lean on NLTK's `TweetTokenizer` rather than sklearn's default token pattern, which throws away too much.

In [1]:
import numpy as np
import pandas as pd
import nltk
nltk.download("twitter_samples", quiet=True)

from nltk.corpus import twitter_samples
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(2024)


In [2]:
pos = twitter_samples.strings("positive_tweets.json")
neg = twitter_samples.strings("negative_tweets.json")
print(f"pos: {len(pos)}  neg: {len(neg)}")
pos[:2]


pos: 5000  neg: 5000


['#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)',
 '@Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks!']

In [3]:
texts = pos + neg
labels = [1] * len(pos) + [0] * len(neg)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=2024
)

# preserve hashtags and emoticons; reduce noisy username repetition
tknzr = TweetTokenizer(strip_handles=True, reduce_len=True)
def tweet_tok(s):
    return tknzr.tokenize(s)

vec = TfidfVectorizer(tokenizer=tweet_tok, ngram_range=(1, 2), min_df=3)
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)
print(f"vocab size: {len(vec.vocabulary_)}")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


vocab size: 7244


In [4]:
clf = LogisticRegression(max_iter=2000, C=4.0)
clf.fit(Xtr, y_train)
pred = clf.predict(Xte)
print(f"accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, target_names=["neg", "pos"]))


accuracy: 1.000
              precision    recall  f1-score   support

         neg       1.00      1.00      1.00      1000
         pos       1.00      1.00      1.00      1000

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



## Quick look at the model's strongest features

In [5]:
feat = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print("most positive:", list(feat[order[-15:]]))
print("most negative:", list(feat[order[:15]]))


most positive: ['happy', ': )', 'your', '!', '! :d', ':) )', '? :)', 'thanks', '. :)', '! :)', ')', ':p', ':d', ':-)', ':)']
most negative: [':(', ':-(', '(', '>:(', '. :(', ':( (', '! :(', '? :(', 'miss', ': (', '( (', '... :(', 'sorry', 'sad', ':( i']


## Notes

Things worth flagging:

- The NLTK twitter dataset is pretty old (around 2015 era) and the language has drifted. The model will not generalize cleanly to current Twitter / X content.
- Class is balanced 50/50 which is unrealistic in the wild.
- I'm not doing any URL or @-handle masking beyond what TweetTokenizer's `strip_handles` does. That's fine here but for a domain dataset I'd probably normalize URLs to a single token.
